# Week 3 — V1 Data Pipeline: UniNet Event Catalogue

**Project:** UniNet Recommender — Enhanced Hybrid Event Recommendation
**Topic 3 lab:** Data Engineering for AI — pipelines, lineage and workflows
**Selected pathway:** **D — JSON / API export** (single modality). Pathways A, B, C, E and F are not run because the project does not use PDFs, standalone CSV tables, images, audio or time-series as a data source this week.

### Why this data (Step A1 — from Topic 2 requirements to data needs)

| Requirement ID | Needed information / feature | Candidate source | Why usable / constraint |
|---|---|---|---|
| **FR2** — rank new events by the meaning of their title and description (Malay and English) | Event title + readable description text for every live event, including events with zero interactions | UniNet `events` collection, read through the public Firestore REST list endpoint (the same documents the public event pages show) | Authoritative (it *is* the live catalogue), full coverage of listed events, public read allowed by `firestore.rules` (`allow read: if true`). Descriptions are stored as Lexical editor JSON, so they must be flattened before any embedding step. |
| **RAI3** — profiles and training data hold no personal identifiers | Event records **without** the poster's user ID, personal name or photo, and without the list of user IDs who reported the event | Same source; personal fields are dropped during preparation | The raw documents carry `userId`, `organizer`, `organizerPhoto` and `reportedBy`. The raw snapshot therefore stays out of the public repository; only the prepared, identifier-free export is committed. |
| **RAI2** — exposure not concentrated on popular clubs or English-language events | Club identity, club-vs-member host, event type/category, and language evidence | Same source (`clubId`, `clubName`, `advancedFilters`) | Club and type fields are present; language is not stored, so it can only be estimated (inspection heuristic below). Language detection is carried to Topic 4. |

The interaction & impression log (needed later for NFR2 learned ranking) is **not** ingested this week: it is private and the impression logging it needs (ADR4) does not exist yet. This pipeline therefore prepares the **Event Catalogue** component of the V1 architecture.

### How to run
- **Google Colab:** run the cells top to bottom. Cell 1 detects Colab and uses `/content/week3_project`. Cell 2 downloads a fresh snapshot of the public catalogue (no API key needed).
- **Locally (from `notebooks/`):** the same cells write straight into the repository's `data/` folder. `data/raw/` is git-ignored because the raw documents contain personal identifiers.

Execution order (lab): Cell 1 → Cell 2 (ingest) → Cell 3 → A3 inspection → Cell 4 → Cell 4A → Cell 5D → B3 → B4.1 → Cell 6A → Cell 6B → Cell 7 → Cell 8D → Cell 9.

## Part A — Start with the Data

### Cell 1 — Workspace setup (RUN FIRST)
Adapted from the lab: the paths are the same in Colab, and point at the repository when the notebook runs locally, so the outputs land where the deliverable summary expects them (`data/processed/`, `data/week3_manifest.json`, `data/pipeline_v1/`).

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_DIR = Path("/content/week3_project")
else:
    # Run from notebooks/: the repository root is one level up.
    BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Running in Colab:", IN_COLAB)
print("Raw folder:", RAW_DIR.relative_to(BASE_DIR))
print("Processed folder:", PROCESSED_DIR.relative_to(BASE_DIR))

Running in Colab: False
Raw folder: data/raw
Processed folder: data/processed


### Cell 2 — Ingest: snapshot the public UniNet event catalogue
This replaces the optional dummy-ZIP cell. It is the **INGEST** stage for the real project source.

- **Source:** Firestore REST list endpoint for the `events` collection of project `uninet-477707` — the same public documents the UniNet event pages read. No API key or login is used.
- **Access:** public read under the platform's Firestore rules. The snapshot is used only inside this project; it is not redistributed (raw stays git-ignored).
- **Behaviour:** pages through the collection, checks every response, and only replaces the saved snapshot once the whole download succeeded (failure point F1). If a snapshot already exists it is reused, so reruns are repeatable; set `REFRESH_SNAPSHOT = True` to take a new one.
- **Raw format:** pages are concatenated into one file; each document is kept exactly as Firestore returned it. A small header records the endpoint and fetch time.

In [2]:
# Cell 2 — Ingest the public UniNet event catalogue (project-specific)
import json
from datetime import datetime, timezone

import requests

SOURCE_ENDPOINT = (
    "https://firestore.googleapis.com/v1/projects/uninet-477707/"
    "databases/(default)/documents/events"
)
SNAPSHOT_PATH = RAW_DIR / "uninet_events" / "events_snapshot.json"
REFRESH_SNAPSHOT = False   # True = download a new snapshot
PAGE_SIZE = 300            # Firestore REST maximum
MAX_PAGES = 20             # hard stop if the collection grows unexpectedly


def fetch_catalogue():
    documents, token, pages = [], None, 0
    while True:
        params = {"pageSize": PAGE_SIZE}
        if token:
            params["pageToken"] = token
        response = requests.get(SOURCE_ENDPOINT, params=params, timeout=30)
        if response.status_code != 200:
            raise RuntimeError(
                f"Catalogue request failed on page {pages + 1}: HTTP {response.status_code}"
            )
        body = response.json()
        page_docs = body.get("documents", [])
        if not isinstance(page_docs, list):
            raise ValueError("Unexpected response shape: 'documents' is not a list")
        documents.extend(page_docs)
        pages += 1
        token = body.get("nextPageToken")
        if not token:
            break
        if pages >= MAX_PAGES:
            raise RuntimeError("Stopped after MAX_PAGES; the snapshot would be partial")
    if not documents:
        raise ValueError("The catalogue returned no documents")
    return {
        "source_endpoint": SOURCE_ENDPOINT,
        "source_collection": "events",
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "access": "public read (firestore.rules: allow read: if true); no credentials used",
        "pages": pages,
        "document_count": len(documents),
        "documents": documents,
    }


if SNAPSHOT_PATH.is_file() and not REFRESH_SNAPSHOT:
    print("Reusing existing snapshot:", SNAPSHOT_PATH.relative_to(BASE_DIR))
else:
    snapshot = fetch_catalogue()
    SNAPSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = SNAPSHOT_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(snapshot, ensure_ascii=False), encoding="utf-8")
    tmp.replace(SNAPSHOT_PATH)  # only overwrite once the whole download succeeded
    print("Saved new snapshot:", SNAPSHOT_PATH.relative_to(BASE_DIR))
    print("Pages:", snapshot["pages"], "| Documents:", snapshot["document_count"])

Reusing existing snapshot: data/raw/uninet_events/events_snapshot.json


### Cell 3 — Verify uploaded / ingested inputs
Same check as the lab (recursive listing), using the paths from Cell 1 instead of re-hard-coding `/content`.

In [3]:
# Cell 3 — Verify inputs
files = sorted(f for f in RAW_DIR.rglob("*") if f.is_file() and f.name != ".gitkeep")
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW_DIR), "bytes:", f.stat().st_size)

assert files, "Ingest or upload at least one relevant input file before proceeding."
print("Inputs verified. Selected input for Step A3: uninet_events/events_snapshot.json")

Input count: 1
uninet_events/events_snapshot.json bytes: 488562
Inputs verified. Selected input for Step A3: uninet_events/events_snapshot.json


### Step A3 — Inspect the selected source (semi-structured JSON)
What the first inspection must prove for semi-structured data: expected keys/schema, nested records, missing keys.
Two representative **subsets** are inspected: **club-hosted** events (have `clubId`) and **member-hosted** events (posted by an individual). Personal fields are **redacted in every printed sample** because this notebook is committed to a public repository.

In [4]:
# A3 — Inspection of the raw catalogue snapshot
import collections
import re

raw_snapshot = json.loads(SNAPSHOT_PATH.read_text(encoding="utf-8"))
raw_docs = raw_snapshot["documents"]

print("--- Source identity ---")
print("Endpoint:", raw_snapshot["source_endpoint"])
print("Fetched at (UTC):", raw_snapshot["fetched_at_utc"])
print("Top-level keys:", list(raw_snapshot.keys()))
print("Documents:", len(raw_docs), "| pages:", raw_snapshot["pages"])
print("Document envelope keys:", sorted(raw_docs[0].keys()))

# Field presence and Firestore value types across all documents
presence = collections.Counter()
types = collections.defaultdict(set)
for doc in raw_docs:
    for key, value in doc.get("fields", {}).items():
        presence[key] += 1
        types[key].update(value.keys())
print("\n--- Field presence (of", len(raw_docs), "documents) ---")
for key, count in presence.most_common():
    print(f"{key:20s} {count:3d}  {sorted(types[key])}")

nested = collections.Counter()
for doc in raw_docs:
    nested.update(doc["fields"].get("advancedFilters", {}).get("mapValue", {}).get("fields", {}).keys())
print("\nNested advancedFilters keys:", dict(nested))

--- Source identity ---
Endpoint: https://firestore.googleapis.com/v1/projects/uninet-477707/databases/(default)/documents/events
Fetched at (UTC): 2026-11-16T01:08:00+00:00
Top-level keys: ['source_endpoint', 'source_collection', 'fetched_at_utc', 'access', 'pages', 'document_count', 'documents']
Documents: 63 | pages: 1
Document envelope keys: ['createTime', 'fields', 'name', 'updateTime']

--- Field presence (of 63 documents) ---
title                 63  ['stringValue']
image                 63  ['stringValue']
organizer             63  ['stringValue']
keywords              63  ['arrayValue']
description           63  ['stringValue']
type                  63  ['stringValue']
reportedBy            63  ['arrayValue']
advancedFilters       63  ['mapValue']
registrationType      63  ['stringValue']
userId                63  ['stringValue']
organizerPhoto        63  ['stringValue']
shareImage            63  ['stringValue']
createdAt             63  ['timestampValue']
expiresAt          

In [5]:
# A3 — Two representative subsets and content checks
PERSONAL_FIELDS = {"userId", "organizer", "organizerPhoto", "reportedBy"}


ALL_UIDS = {d["fields"]["userId"]["stringValue"] for d in raw_docs if "userId" in d["fields"]}


def redacted(doc, max_len=110):
    # Printable copy of a raw document with personal fields hidden
    # (user IDs are also masked where they appear inside other values, e.g. storage URLs).
    out = {"name": doc["name"].split("/")[-1], "updateTime": doc.get("updateTime")}
    for key, value in sorted(doc["fields"].items()):
        text = "<redacted: personal field>" if key in PERSONAL_FIELDS else json.dumps(value, ensure_ascii=False)
        for uid in ALL_UIDS:
            text = text.replace(uid, "<uid>")
        out[key] = text if len(text) <= max_len else text[:max_len] + "..."
    return out


club_docs = [d for d in raw_docs if "clubId" in d["fields"]]
member_docs = [d for d in raw_docs if "clubId" not in d["fields"]]
print("Subset 1 — club-hosted events:", len(club_docs))
print("Subset 2 — member-hosted events:", len(member_docs))

for label, subset in (("Subset 1 sample (club-hosted)", club_docs), ("Subset 2 sample (member-hosted)", member_docs)):
    print(f"\n--- {label} ---")
    for key, value in redacted(subset[0]).items():
        print(f"  {key}: {value}")

# Description format: stored as serialised Lexical editor state?
descriptions = [d["fields"]["description"]["stringValue"] for d in raw_docs]
lexical_count = sum(text.lstrip().startswith("{") for text in descriptions)
print("\n--- Content checks ---")
print("Descriptions stored as Lexical JSON:", lexical_count, "of", len(descriptions))
print("Raw description length (chars) — min/median/max:",
      min(map(len, descriptions)), sorted(map(len, descriptions))[len(descriptions) // 2], max(map(len, descriptions)))

field = lambda d, k: d["fields"].get(k, {})
types_seen = collections.Counter(
    (field(d, "type").get("stringValue"),
     d["fields"]["advancedFilters"]["mapValue"]["fields"].get("category", {}).get("stringValue"))
    for d in raw_docs)
print("(type, category) pairs:", dict(types_seen))
deadlines = [d["fields"]["advancedFilters"]["mapValue"]["fields"].get("registrationDeadline", {}).get("stringValue", "") for d in raw_docs]
print("Empty registration deadlines:", sum(1 for x in deadlines if not x))
universities_set = sum(1 for d in raw_docs if d["fields"]["advancedFilters"]["mapValue"]["fields"].get("universities", {}).get("arrayValue", {}).get("values"))
print("Events with an audience university set:", universities_set)
reports = collections.Counter(len(field(d, "reportedBy").get("arrayValue", {}).get("values", [])) for d in raw_docs)
print("Report counts:", dict(reports), "| unlisted:", sum(1 for d in raw_docs if field(d, "visibility").get("stringValue") == "unlisted"))
created = sorted(field(d, "createdAt")["timestampValue"] for d in raw_docs)
print("createdAt range:", created[0], "->", created[-1])
titles = collections.Counter(field(d, "title")["stringValue"].strip().lower() for d in raw_docs)
print("Duplicate titles:", sum(1 for n in titles.values() if n > 1))

Subset 1 — club-hosted events: 43
Subset 2 — member-hosted events: 20

--- Subset 1 sample (club-hosted) ---
  name: 0OaH7ZiCHtT7ZP6yFedB
  updateTime: 2026-10-22T14:10:38.450421Z
  advancedFilters: {"mapValue": {"fields": {"deadlinePreset": {"nullValue": null}, "universities": {"arrayValue": {}}, "registrat...
  clubId: {"stringValue": "7Kk64tgooC1iKgpLwaip"}
  clubLogo: {"stringValue": "https://firebasestorage.googleapis.com/v0/b/uninet-477707.firebasestorage.app/o/clubs%2Flogos...
  clubName: {"stringValue": "UniNet"}
  clubVerified: {"booleanValue": true}
  commentCount: {"integerValue": "2"}
  createdAt: {"timestampValue": "2026-10-22T14:01:28.960Z"}
  description: {"stringValue": "{\"root\":{\"children\":[{\"children\":[{\"detail\":0,\"format\":0,\"mode\":\"normal\",\"styl...
  expiresAt: {"timestampValue": "2026-11-21T14:01:26.991Z"}
  image: {"stringValue": "https://firebasestorage.googleapis.com/v0/b/uninet-477707.firebasestorage.app/o/events%2F<uid...
  keywords: {"arrayValue

In [6]:
# A3 — Language evidence for RAI2 (heuristic, inspection only — not written to the prepared data)
MALAY_MARKERS = {
    "dan", "untuk", "yang", "akan", "dengan", "kepada", "ini", "pelajar", "anda",
    "peserta", "daripada", "bagi", "sila", "pertandingan", "hadiah", "tarikh", "tempat", "jom",
}


def quick_text(raw):
    # Rough text pull for inspection only; Part B uses the full Lexical flattener.
    return " ".join(re.findall(r'"text":"(.*?)"', raw))


malay_like = 0
for doc in raw_docs:
    words = set(re.findall(r"[a-z]+", (doc["fields"]["title"]["stringValue"] + " " +
                                        quick_text(doc["fields"]["description"]["stringValue"])).lower()))
    if len(words & MALAY_MARKERS) >= 3:
        malay_like += 1
print("Events with >= 3 distinct Malay marker words:", malay_like, "of", len(raw_docs))
print("Limitation: a word-list heuristic, not language detection; mixed Malay-English text is common.")

Events with >= 3 distinct Malay marker words: 5 of 63
Limitation: a word-list heuristic, not language detection; mixed Malay-English text is common.


**A3 observations (recorded in `docs/week3_data_inventory.md`):**
- One JSON source, Firestore REST format: every value is wrapped in a type key (`stringValue`, `timestampValue`, `arrayValue`, `mapValue`, …), and category, flairs, universities and deadline sit inside the nested `advancedFilters` map.
- Every description is serialised Lexical editor state, not text — unusable for FR2 embeddings until flattened.
- Personal fields are present on every document (`userId`, `organizer`, `organizerPhoto`, `reportedBy`) → RAI3 requires removing them before anything is committed or trained on.
- `type = "none"` always pairs with `category = "All Events"` (uncategorised events); some registration deadlines are empty strings; no event sets an audience university; no event is unlisted or report-hidden in this snapshot.
- 43 events are club-hosted across only 4 clubs, and 32 of the 63 events sit under the platform's own "UniNet" club — an exposure-concentration signal to watch for RAI2. 20 events are member-hosted.
- Events expire about 30 days after posting, so the collection only holds recent events (oldest `createdAt` 2026-10-16; one event had already passed `expiresAt` at snapshot time). This snapshot cannot provide history for offline replay (NFR2) — that needs the interaction log later.

## Part B — Prepare the Data

### Cell 4 — REQUIRED: Source selection + lineage utilities (unchanged from the lab)

In [7]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone


def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]


def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target.relative_to(BASE_DIR))
    return manifest


print("Cell 4 completed successfully.")
print("Available functions: select_source(), source_hash(), save_manifest()")

Cell 4 completed successfully.
Available functions: select_source(), source_hash(), save_manifest()


### Cell 4A — REQUIRED: Common lineage registry (unchanged from the lab)

In [8]:
# CELL 4A — COMMON LINEAGE REGISTRY
lineage_entries = {}


def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)


print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


### Cell 5D — CONDITIONAL REQUIRED: Pathway D (JSON / API export), adapted to the UniNet schema
Changes from the lab's starter, each tied to a requirement:

| Setting / step | Value | Why |
|---|---|---|
| `RECORD_KEY` | `"documents"` | Firestore returns records under `documents`, not `items`. |
| `REQUIRED_KEYS` | envelope `name`, `fields`; fields `title`, `description`, `type`, `createdAt` | Minimum needed to identify an event and build FR2 text. Presence only, not value validity. |
| Decode Firestore typed values | `{"stringValue": "x"}` → `"x"`, maps/arrays recursively; unknown type → error | Downstream code needs plain values; an unknown type means the schema changed (stop, do not guess). |
| Flatten Lexical description | Same walk as UniNet's `netlify/shared/lexical.ts` | FR2 needs readable text; parity means the embedded text is exactly what users and crawlers see. |
| Mask contact details | Phone numbers → `[phone]`, emails → `[email]` inside the description text | RAI3 data minimisation: descriptions sometimes name a contact person with a phone number; contacts carry no ranking signal. |
| Public-field whitelist | Drop `userId`, `organizer`, `organizerPhoto`, `reportedBy`; also drop image/QR/registration URLs and title-derived `keywords` | RAI3 (no personal identifiers) and scope (URLs are not ranking inputs). |
| Exclusion rule | Drop `visibility == "unlisted"` or ≥ 5 reports | FR1: unlisted / moderated events must never be recommended. Same threshold as the platform. |
| Light normalisation only | Trim title; empty deadline string → `null`; add `host_kind`, `embedding_text` | No imputation, no relabelling: `type`/`category` values are kept as stored and their quirks go to Topic 4. |
| Output order | Sorted by `created_at`, then `event_id` | Deterministic bytes, so the Cell 9 repeatability check is meaningful. |

In [9]:
# Cell 5D — PATHWAY D: JSON/API EXPORT (UniNet event catalogue)
import json
import re

# Step 1: Select the exact JSON source
f = select_source(relative_path="uninet_events/events_snapshot.json")

# Step 2: Load the JSON source
with f.open("r", encoding="utf-8") as file:
    raw = json.load(file)

# Step 3: Configure record extraction and required fields
RECORD_KEY = "documents"
REQUIRED_KEYS = ["name", "fields"]
REQUIRED_FIELDS = ["title", "description", "type", "createdAt"]
PERSONAL_FIELDS = ["userId", "organizer", "organizerPhoto", "reportedBy"]
MAX_REPORTS = 5            # matches the platform's report-hiding threshold
PROCESSING_LABEL = "decode-firestore + lexical-flatten + mask-contacts + public-whitelist + exclude-unlisted-reported"
PHONE_PATTERN = re.compile(r"(?:\+?60|0)1\d[\s-]?\d{3,4}[\s-]?\d{4}")   # Malaysian mobile numbers
EMAIL_PATTERN = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")


# Step 4: Transformation helpers (reused unchanged by Cell 8D)
def fs_value(value):
    # Decode one Firestore REST typed value into plain Python.
    if "stringValue" in value:
        return value["stringValue"]
    if "integerValue" in value:
        return int(value["integerValue"])
    if "doubleValue" in value:
        return float(value["doubleValue"])
    if "booleanValue" in value:
        return bool(value["booleanValue"])
    if "nullValue" in value:
        return None
    if "timestampValue" in value:
        return value["timestampValue"]
    if "arrayValue" in value:
        return [fs_value(v) for v in value["arrayValue"].get("values", [])]
    if "mapValue" in value:
        return {k: fs_value(v) for k, v in value["mapValue"].get("fields", {}).items()}
    raise ValueError(f"Unknown Firestore value type: {list(value)}")


def lexical_to_text(raw_state):
    # Port of UniNet netlify/shared/lexical.ts: flatten Lexical editor state to plain text.
    try:
        parsed = json.loads(raw_state)
    except (TypeError, ValueError):
        return ""
    parts = []

    def walk(node):
        if not isinstance(node, dict):
            return
        if isinstance(node.get("text"), str):
            parts.append(node["text"])
        if isinstance(node.get("children"), list):
            for child in node["children"]:
                walk(child)
        if node.get("type") in ("paragraph", "listitem", "heading"):
            parts.append(" ")

    root = parsed.get("root") if isinstance(parsed, dict) else None
    walk(root if root is not None else parsed)
    return re.sub(r"\s+", " ", "".join(parts)).strip()


def description_text(raw_description):
    # Lexical state is flattened; legacy plain text has markup stripped (same rule as UniNet).
    source = raw_description or ""
    if source.lstrip().startswith("{"):
        return lexical_to_text(source)
    return re.sub(r"\s+", " ", re.sub(r"<[^>]*>?", "", source)).strip()


def mask_contacts(text):
    # RAI3 data minimisation: contact details carry no ranking signal, so they are masked.
    return EMAIL_PATTERN.sub("[email]", PHONE_PATTERN.sub("[phone]", text))


def extract_records(obj):
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict) and isinstance(obj.get(RECORD_KEY), list):
        return obj[RECORD_KEY]
    raise ValueError(f"Expected a JSON list or an object with a list under '{RECORD_KEY}'.")


def is_valid(doc):
    return (isinstance(doc, dict)
            and all(key in doc for key in REQUIRED_KEYS)
            and all(name in doc["fields"] for name in REQUIRED_FIELDS))


def exclusion_reason(fields):
    if fields.get("visibility") == "unlisted":
        return "unlisted"
    if len(fields.get("reportedBy") or []) >= MAX_REPORTS:
        return "report-hidden"
    return None


def to_prepared(doc, source_rel, source_sha):
    fields = {k: fs_value(v) for k, v in doc["fields"].items()}
    filters = fields.get("advancedFilters") or {}
    title = fields["title"].strip()
    text = mask_contacts(description_text(fields["description"]))
    club_id = fields.get("clubId")
    return {
        "record_id": f"uninet-event-{doc['name'].split('/')[-1]}",
        "event_id": doc["name"].split("/")[-1],
        "title": title,
        "description_text": text,
        "embedding_text": f"{title}\n\n{text}" if text else title,
        "description_empty": not text,
        "type": fields.get("type"),
        "category": filters.get("category"),
        "flairs": filters.get("flairs") or [],
        "universities": filters.get("universities") or [],
        "host_kind": "club" if club_id else "member",
        "club_id": club_id,
        "club_name": fields.get("clubName"),
        "club_verified": fields.get("clubVerified"),
        "registration_type": fields.get("registrationType"),
        "registration_deadline": filters.get("registrationDeadline") or None,
        "created_at": fields["createdAt"],
        "expires_at": fields.get("expiresAt"),
        "comment_count": fields.get("commentCount"),
        "source": source_rel,
        "source_sha256": source_sha,
        "firestore_update_time": doc.get("updateTime"),
        "processing": PROCESSING_LABEL,
    }


def prepare_catalogue(obj, source_rel, source_sha):
    # Returns (prepared records, summary counts). Pure function of the input.
    records = extract_records(obj)
    valid = [d for d in records if is_valid(d)]
    kept, excluded = [], collections.Counter()
    for doc in valid:
        reason = exclusion_reason({k: fs_value(v) for k, v in doc["fields"].items()})
        if reason:
            excluded[reason] += 1
        else:
            kept.append(to_prepared(doc, source_rel, source_sha))
    kept.sort(key=lambda r: (r["created_at"], r["event_id"]))
    summary = {
        "original_records": len(records),
        "valid_records": len(valid),
        "missing_required": len(records) - len(valid),
        "excluded": dict(excluded),
        "prepared_records": len(kept),
        "records_with_masked_contacts": sum(
            1 for r in kept if "[phone]" in r["description_text"] or "[email]" in r["description_text"]),
    }
    return kept, summary


# Step 5: Inspect record structure
items = extract_records(raw)
print("--- Original JSON Data ---")
print("Source:", f.relative_to(RAW_DIR))
print("Top-level format:", type(raw).__name__, "| record key:", RECORD_KEY)
print("Record count:", len(items))
print("First record envelope keys:", sorted(items[0].keys()))

# Step 6: Prepare, validate required fields and apply exclusions
SOURCE_REL = str(f.relative_to(RAW_DIR))
SOURCE_SHA = source_hash(f)
prepared_records, prep_summary = prepare_catalogue(raw, SOURCE_REL, SOURCE_SHA)
print("\n--- Validation Summary ---")
for key, value in prep_summary.items():
    print(f"{key}: {value}")
assert prepared_records, "No valid JSON records remain after validation."

# Step 7: Export prepared records to JSONL
prepared_path = PROCESSED_DIR / "prepared_json.jsonl"
with prepared_path.open("w", encoding="utf-8") as output:
    for row in prepared_records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

# Step 8: Validate the exported JSONL
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
assert exported_records == prepared_records, "Exported content differs from prepared records"

print("\nPATHWAY D completed successfully.")
print("Exported records:", len(exported_records))
print("Saved:", prepared_path.relative_to(BASE_DIR))
print("Validation: PASS")

--- Original JSON Data ---
Source: uninet_events/events_snapshot.json
Top-level format: dict | record key: documents
Record count: 63
First record envelope keys: ['createTime', 'fields', 'name', 'updateTime']

--- Validation Summary ---
original_records: 63
valid_records: 63
missing_required: 0
excluded: {}
prepared_records: 63
records_with_masked_contacts: 5

PATHWAY D completed successfully.
Exported records: 63
Saved: data/processed/prepared_json.jsonl
Validation: PASS


### Step B3 — Validate the prepared result and compare one decision
**Decision compared:** how much of the raw document to carry forward and in what form.
- **Configuration A — decode only:** decode Firestore typed values and keep every field as stored (description stays Lexical JSON, personal fields stay).
- **Configuration B — selected:** decode + flatten Lexical + public-field whitelist + exclusions (Cell 5D).

Both run on the same snapshot.

In [10]:
# B3 — Compare configuration A (decode only) with configuration B (selected)
import statistics

config_a = [{k: fs_value(v) for k, v in d["fields"].items()} for d in extract_records(raw) if is_valid(d)]
config_b = prepared_records

a_desc = [len(r["description"]) for r in config_a]
b_desc = [len(r["description_text"]) for r in config_b]
a_bytes = sum(len(json.dumps(r, ensure_ascii=False)) for r in config_a)
b_bytes = sum(len(json.dumps(r, ensure_ascii=False)) for r in config_b)

comparison = {
    "records": (len(config_a), len(config_b)),
    "embedding-ready text field": ("no", "yes (embedding_text)"),
    "median description chars": (statistics.median(a_desc), statistics.median(b_desc)),
    "description is readable prose": ("no (Lexical JSON)", "yes"),
    "records with contact details in text": (
        sum(1 for r in config_a if PHONE_PATTERN.search(description_text(r["description"])) or EMAIL_PATTERN.search(description_text(r["description"]))),
        sum(1 for r in config_b if PHONE_PATTERN.search(r["description_text"]) or EMAIL_PATTERN.search(r["description_text"])),
    ),
    "records with personal fields": (
        sum(1 for r in config_a if any(k in r for k in PERSONAL_FIELDS)),
        sum(1 for r in config_b if any(k in r for k in PERSONAL_FIELDS)),
    ),
    "serialised size (KB)": (round(a_bytes / 1024, 1), round(b_bytes / 1024, 1)),
}
print(f"{'measure':38s} {'A: decode only':>20s} {'B: selected':>22s}")
for measure, (a_value, b_value) in comparison.items():
    print(f"{measure:38s} {str(a_value):>20s} {str(b_value):>22s}")

sample = config_b[0]
print("\nSame event, description start:")
print("  A:", repr(config_a[[r['title'].strip() for r in config_a].index(sample['title'])]["description"][:150]))
print("  B:", repr(sample["description_text"][:150]))

measure                                      A: decode only            B: selected
records                                                  63                     63
embedding-ready text field                               no   yes (embedding_text)
median description chars                               4180                    847
description is readable prose             no (Lexical JSON)                    yes
records with contact details in text                      5                      0
records with personal fields                             63                      0
serialised size (KB)                                  421.1                  180.4

Same event, description start:
  A: '{"root":{"children":[{"children":[{"detail":0,"format":0,"mode":"normal","style":"","text":"Hey everyone! 👋 Here\'s an opportunity for students from Ma'
  B: "Hey everyone! 👋 Here's an opportunity for students from Malaysian public universities, polytechnics, and community colleges to participate 

**B3 — basic output contract (lab) plus two project-specific contracts.** RAI3 is checked twice: no personal field names, and no raw user ID string anywhere in the output (catches a leak through any other field).

In [11]:
# B3 BASIC OUTPUT CONTRACT — plus RAI3 and FR2 contracts
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
print("Output:", prepared_path.relative_to(BASE_DIR), "Size:", prepared_path.stat().st_size)

exported_text = prepared_path.read_text(encoding="utf-8")
raw_user_ids = {d["fields"]["userId"]["stringValue"] for d in raw_docs if "userId" in d["fields"]}
leaked_fields = [k for r in exported_records for k in PERSONAL_FIELDS if k in r]
leaked_ids = [uid for uid in raw_user_ids if uid in exported_text]
assert not leaked_fields, f"Personal fields in output: {set(leaked_fields)}"
assert not leaked_ids, f"{len(leaked_ids)} raw user IDs found in output"
assert not PHONE_PATTERN.search(exported_text) and not EMAIL_PATTERN.search(exported_text), "Contact detail in output"
print("RAI3 contract: 0 personal fields, 0 of", len(raw_user_ids), "raw user IDs, 0 phone/email patterns in output — PASS")

empty = [r["event_id"] for r in exported_records if r["description_empty"]]
assert all(r["embedding_text"].strip() for r in exported_records), "Empty embedding_text"
print("FR2 contract: every record has embedding_text; records with empty description:", len(empty))
print("Validation: file exists and is non-empty; representative record below for manual check.")

show = {k: (v[:160] + "..." if isinstance(v, str) and len(v) > 160 else v) for k, v in exported_records[0].items()}
print(json.dumps(show, indent=2, ensure_ascii=False))

Output: data/processed/prepared_json.jsonl Size: 196075
RAI3 contract: 0 personal fields, 0 of 12 raw user IDs, 0 phone/email patterns in output — PASS
FR2 contract: every record has embedding_text; records with empty description: 0
Validation: file exists and is non-empty; representative record below for manual check.
{
  "record_id": "uninet-event-zZCfESCxZ1zIYVlRieSy",
  "event_id": "zZCfESCxZ1zIYVlRieSy",
  "title": "Gemini NO-CODE Hackathon 2026: Google for Education",
  "description_text": "Hey everyone! 👋 Here's an opportunity for students from Malaysian public universities, polytechnics, and community colleges to participate in a hackathon! 💡 Wha...",
  "embedding_text": "Gemini NO-CODE Hackathon 2026: Google for Education\n\nHey everyone! 👋 Here's an opportunity for students from Malaysian public universities, polytechnics, and co...",
  "description_empty": false,
  "type": "competition",
  "category": "Competition",
  "flairs": [
    "Hackathon"
  ],
  "universities": [],
  

**B3 decision:** configuration **B**. *Benefit:* readable, embedding-ready text for every event (FR2) and zero personal identifiers or contact details in anything committed or used for training (RAI3), at under half the size. *Accepted limitation:* flattening drops the description's paragraph and list structure and any emphasis, and the poster image (which sometimes holds the event date) is not used — acceptable because FR2 embeds short event text, and the structure can be revisited if retrieval quality suffers.

### Step B4.1 — Register Pathway D lineage

In [12]:
# Register Pathway D — JSON/API
register_lineage(
    source=f,
    output=prepared_path,
    processing=(
        f"Firestore REST JSON to JSONL; record_key={RECORD_KEY}; "
        f"required_keys={REQUIRED_KEYS}+fields{REQUIRED_FIELDS}; {PROCESSING_LABEL}; "
        f"dropped personal fields {PERSONAL_FIELDS}; excluded={prep_summary['excluded'] or 'none'}"
    ),
)

Registered: events_snapshot.json -> prepared_json.jsonl


### Cell 6A — Generate the common lineage manifest (source version and access filled from the snapshot instead of 'Not verified')

In [13]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
assert lineage_entries, "No lineage entries registered."
lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source, output, processing = entry["source"], entry["output"], entry["processing"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": f"Firestore snapshot fetched {raw_snapshot['fetched_at_utc']} "
                          f"({raw_snapshot['document_count']} documents; per-record updateTime kept)",
        "access_rights": "Public read of UniNet events collection; raw snapshot contains personal "
                         "identifiers and is NOT committed; prepared output is identifier-free",
        "status": "Prepared; quality review pending (Topic 4)",
    })

manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))

Saved lineage: data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 1


### Cell 6B — Verify the common lineage manifest (unchanged from the lab)

In [14]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."
saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path.relative_to(BASE_DIR))

COMMON LINEAGE VALIDATION: PASS
Verified records: 1
Manifest: data/week3_manifest.json


## Part C — Build and Document the V1 Data Pipeline

### Cell 7 — REQUIRED: Common V1 pipeline utilities
Same logic as the lab, with two documented adaptations: it reuses the Cell 1 paths instead of re-hard-coding `/content`, and it writes paths **relative to the project root** into the run manifest, so the manifest is identical in Colab and locally and does not expose machine paths.

In [15]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}


def rel(path):
    return str(Path(path).resolve().relative_to(BASE_DIR.resolve()))


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    elif not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    return source


def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)
    # INGEST
    if kind == "images":
        inputs = sorted(p for p in source.iterdir()
                        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
    else:
        inputs = [source]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")
    # INSPECT
    inspection = inspect_fn(inputs, configuration)
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    # PREPARE / STORE
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    # VALIDATE
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")
    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs],
    }
    # RECORD LINEAGE
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": rel(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [rel(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": inspection.get("source_version", "Not verified"),
        "access_rights": "Public read of UniNet events; raw snapshot not committed (contains personal identifiers)",
    }
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", rel(manifest_path))
    return record


def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first


print("Cell 7 ready. Run ONLY the selected Cell 8 pathway(s).")

Cell 7 ready. Run ONLY the selected Cell 8 pathway(s).


### Cell 8D — CONDITIONAL REQUIRED: JSON / API export pipeline
Uses the **same source and configuration as Part B**: the preparation calls the Cell 5D functions (`prepare_catalogue`, `fs_value`, `lexical_to_text`), so Part B and Part C cannot drift apart. The `prepare` stage also enforces the RAI3 contract and refuses to write output that contains a personal field or raw user ID (failure point F2).

In [16]:
# Cell 8D — CONDITIONAL REQUIRED: JSON/API Projects (UniNet event catalogue)
SOURCE_D = resolve_source("uninet_events/events_snapshot.json")

CONFIG_D = {
    "record_key": RECORD_KEY,
    "required_keys": REQUIRED_KEYS,
    "required_fields": REQUIRED_FIELDS,
    "drop_personal_fields": PERSONAL_FIELDS,
    "exclude": {"visibility": "unlisted", "min_reports": MAX_REPORTS},
    "description": "lexical-flatten (UniNet lexical.ts parity) + mask phone/email",
    "invalid_record_policy": "exclude",
    "sort": ["created_at", "event_id"],
}


def load_snapshot(src):
    with Path(src).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def inspect_json(inputs, cfg):
    obj = load_snapshot(inputs[0])
    records = extract_records(obj)
    valid = [d for d in records if is_valid(d)]
    return {
        "source_version": f"snapshot fetched {obj.get('fetched_at_utc')}",
        "record_count": len(records),
        "valid_records": len(valid),
        "missing_required": len(records) - len(valid),
        "club_hosted": sum(1 for d in valid if "clubId" in d["fields"]),
        "member_hosted": sum(1 for d in valid if "clubId" not in d["fields"]),
        "lexical_descriptions": sum(1 for d in valid if d["fields"]["description"]["stringValue"].lstrip().startswith("{")),
        "records_with_personal_fields": sum(1 for d in valid if any(k in d["fields"] for k in cfg["drop_personal_fields"])),
    }


def prepare_json(inputs, out, cfg):
    src = inputs[0]
    obj = load_snapshot(src)
    if cfg["invalid_record_policy"] != "exclude":
        raise ValueError("This V1 pipeline supports only the 'exclude' policy.")
    records, summary = prepare_catalogue(obj, str(src.relative_to(RAW_DIR)), sha256_file(src))
    if not records:
        raise ValueError("No valid JSON records remain after validation.")

    # RAI3 contract: refuse to store anything carrying a personal identifier
    lines = [json.dumps(r, ensure_ascii=False) for r in records]
    blob = "\n".join(lines)
    user_ids = {d["fields"]["userId"]["stringValue"] for d in extract_records(obj) if "userId" in d.get("fields", {})}
    if (any(k in r for r in records for k in cfg["drop_personal_fields"]) or any(uid in blob for uid in user_ids)
            or PHONE_PATTERN.search(blob) or EMAIL_PATTERN.search(blob)):
        raise ValueError("Personal identifier found in prepared output; export stopped.")

    target = out / "prepared_json.jsonl"
    target.write_text(blob + "\n", encoding="utf-8")

    with target.open("r", encoding="utf-8") as handle:
        exported = [json.loads(line) for line in handle if line.strip()]
    if exported != records:
        raise ValueError("Exported JSONL content does not match prepared records.")

    mapping = [{"source": rel(src), "output": rel(target)}]
    method = (f"Firestore REST JSON to JSONL; {PROCESSING_LABEL}; "
              f"prepared={summary['prepared_records']}; excluded={summary['excluded'] or 'none'}; "
              f"missing_required={summary['missing_required']}")
    return [target], mapping, method


run_D = register_and_run("json", SOURCE_D, inspect_json, prepare_json, CONFIG_D)
print("Cell 8D completed successfully.")
print("Inspection:", run_D["inspection"])

SUCCESS: json | Inputs: 1 | Outputs: 1
Manifest: data/pipeline_v1/json/pipeline_manifest.json
Cell 8D completed successfully.
Inspection: {'source_version': 'snapshot fetched 2026-11-16T01:08:00+00:00', 'record_count': 63, 'valid_records': 63, 'missing_required': 0, 'club_hosted': 43, 'member_hosted': 20, 'lexical_descriptions': 63, 'records_with_personal_fields': 63}


### Cell 9 — REQUIRED: Repeatability and provenance checks (unchanged from the lab)

In [17]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
assert PIPELINE_RUNS, "No pipeline runs registered. Run at least one selected Cell 8 pathway first."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Mapping changed for {kind}"
    print(f"PASS: {kind} — source fingerprints, prepared outputs and mappings match")

# Part B and Part C must agree: same source and configuration -> same bytes
part_b = sha256_file(PROCESSED_DIR / "prepared_json.jsonl")
part_c = PIPELINE_RUNS["json"]["first"]["validation"]["output_sha256"][0]
assert part_b == part_c, "Part B and Part C outputs differ"
print("PASS: Part B (data/processed) and Part C (data/pipeline_v1/json) outputs are byte-identical")
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---


SUCCESS: json | Inputs: 1 | Outputs: 1
Manifest: data/pipeline_v1/json/pipeline_manifest.json
PASS: json — source fingerprints, prepared outputs and mappings match
PASS: Part B (data/processed) and Part C (data/pipeline_v1/json) outputs are byte-identical

All selected pipelines passed repeatability smoke checks.


### Expected evidence — checklist
- `SUCCESS: json | Inputs: 1 | Outputs: 1` and a `pipeline_manifest.json` in `data/pipeline_v1/json/`.
- `PASS` repeatability messages, and Part B / Part C outputs byte-identical.
- Representative prepared record inspected in B3 (readable `description_text`, no personal fields).
- What this does **not** prove: data quality or fitness for purpose (Topic 4), freshness beyond the snapshot time, or rights to redistribute the raw documents.

Workflow diagram: `docs/week3_pipeline_diagram.png` · Decisions and stage contracts: `docs/week3_pipeline_decisions.md` · Inventory: `docs/week3_data_inventory.md`.